# boolean-mask-identity-replace — worked example 2: Zero out flagged columns of a 2-D tensor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-identity-replace`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

A 1-D bool mask can index a single axis of a higher-rank tensor. To zero whole columns of a `(R, C)` tensor you index the second axis: `out[:, col_mask] = 0.0`. The scalar broadcasts to fill every selected column, and unselected columns stay untouched.

## Worked solution

Given a `(R, C)` tensor `m` and a `(C,)` bool mask flagging columns to drop, we want a copy with those columns zeroed.

1. **Clone.** `out = m.clone()` so the caller's `m` is never mutated.
2. **Index the column axis.** `out[:, mask]` uses `:` to keep every row and the 1-D bool `mask` to pick the flagged columns. This selects a `(R, K)` block where `K = mask.sum()`.
3. **Broadcast-assign zero.** `out[:, mask] = 0.0` writes the scalar `0.0` across that whole block in one assignment. The placement of `mask` on the **second** index slot is what targets columns rather than rows — contrast `out[mask]` which would target rows.
4. **Return `out`.** Columns where `mask` is `False` are byte-for-byte unchanged.

In [ ]:
def zero_flagged_cols(m: Tensor, mask: Tensor) -> Tensor:
    out = m.clone()
    out[:, mask] = 0.0
    return out

t.manual_seed(0)
m = t.randn(3, 4)
mask = t.tensor([True, False, True, False])
out = zero_flagged_cols(m, mask)
print('flagged cols zero    :', bool((out[:, mask] == 0).all()))
print('unflagged cols kept  :', bool(t.equal(out[:, ~mask], m[:, ~mask])))
print('input not mutated    :', out.shape == m.shape)